In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('data/train.csv')

In [3]:
df.head()

,Id,Filename,Label
0,Actor_02_03-01-03-02-02-01-02,03-01-03-02-02-01-02.wav,happy
1,Actor_07_03-01-07-01-02-01-07,03-01-07-01-02-01-07.wav,disgust
2,Actor_09_03-01-08-02-01-01-09,03-01-08-02-01-01-09.wav,surprised
3,Actor_13_03-01-08-02-02-01-13,03-01-08-02-02-01-13.wav,surprised
4,Actor_18_03-01-05-02-02-02-18,03-01-05-02-02-02-18.wav,angry


In [4]:
df['Filepath'] = df['Filename'].apply(lambda x : 'data\Train\\'+ x)
df.head()

,Id,Filename,Label,Filepath
0,Actor_02_03-01-03-02-02-01-02,03-01-03-02-02-01-02.wav,happy,data\Train\03-01-03-02-02-01-02.wav
1,Actor_07_03-01-07-01-02-01-07,03-01-07-01-02-01-07.wav,disgust,data\Train\03-01-07-01-02-01-07.wav
2,Actor_09_03-01-08-02-01-01-09,03-01-08-02-01-01-09.wav,surprised,data\Train\03-01-08-02-01-01-09.wav
3,Actor_13_03-01-08-02-02-01-13,03-01-08-02-02-01-13.wav,surprised,data\Train\03-01-08-02-02-01-13.wav
4,Actor_18_03-01-05-02-02-02-18,03-01-05-02-02-02-18.wav,angry,data\Train\03-01-05-02-02-02-18.wav


In [5]:
label_map = {
    'neutral':1,
    'calm':2,
    'happy':3,
    'sad':4,
    'angry':5,
    'fearful':6,
    'disgust':7,
    'surprised':8
}

df['label_id'] = df['Label'].map(label_map)
df.head()

,Id,Filename,Label,Filepath,label_id
0,Actor_02_03-01-03-02-02-01-02,03-01-03-02-02-01-02.wav,happy,data\Train\03-01-03-02-02-01-02.wav,3
1,Actor_07_03-01-07-01-02-01-07,03-01-07-01-02-01-07.wav,disgust,data\Train\03-01-07-01-02-01-07.wav,7
2,Actor_09_03-01-08-02-01-01-09,03-01-08-02-01-01-09.wav,surprised,data\Train\03-01-08-02-01-01-09.wav,8
3,Actor_13_03-01-08-02-02-01-13,03-01-08-02-02-01-13.wav,surprised,data\Train\03-01-08-02-02-01-13.wav,8
4,Actor_18_03-01-05-02-02-02-18,03-01-05-02-02-02-18.wav,angry,data\Train\03-01-05-02-02-02-18.wav,5


In [6]:
dataset = df[['Filepath', 'label_id']]
dataset

,Filepath,label_id
0,data\Train\03-01-03-02-02-01-02.wav,3
1,data\Train\03-01-07-01-02-01-07.wav,7
2,data\Train\03-01-08-02-01-01-09.wav,8
3,data\Train\03-01-08-02-02-01-13.wav,8
4,data\Train\03-01-05-02-02-02-18.wav,5
...,...,...
1147,data\Train\03-01-01-01-01-02-09.wav,1
1148,data\Train\03-01-03-01-02-02-10.wav,3
1149,data\Train\03-01-04-01-01-02-21.wav,4
1150,data\Train\03-01-05-02-01-02-07.wav,5


In [7]:
from sklearn.model_selection import train_test_split

In [8]:
df_train, df_test  = train_test_split(dataset, test_size= 0.2, random_state= 42, stratify= dataset['label_id'])
print("Train:", df_train.shape, "Test:", df_test.shape)
print("Label % (train):\n", df_train["label_id"].value_counts(normalize=False).sort_index())
print("Label % (test):\n",  df_test["label_id"].value_counts(normalize=True).sort_index())

Train: (921, 2) Test: (231, 2)
Label % (train):
 label_id
1     62
2    123
3    123
4    123
5    123
6    122
7    123
8    122
Name: count, dtype: int64
Label % (test):
 label_id
1    0.064935
2    0.134199
3    0.129870
4    0.134199
5    0.134199
6    0.134199
7    0.134199
8    0.134199
Name: proportion, dtype: float64


In [9]:
import soundfile as sf
import torch
import torchaudio

In [10]:
row = df_train.iloc[0]
filepath = row['Filepath']

audio, sample_rate = sf.read(filepath)
if audio.ndim == 2:
    audio = audio.mean(axis=1)
waveform = torch.tensor(audio, dtype= torch.float32).unsqueeze(0)


print("Waveform shape:", waveform.shape)   # [channels, time]
print("Sample rate:", sample_rate)

Waveform shape: torch.Size([1, 185785])
Sample rate: 48000


In [11]:
def extract_waveform(filepath):
    audio, sr = sf.read(filepath)
    if audio.ndim == 2:
        audio = audio.mean(axis=1)
    waveform = torch.tensor(audio, dtype= torch.float32).unsqueeze(0)
    return waveform, sr

In [12]:
def resample_waveform(sr, target_sr, waveform):
    if sr != target_sr:
        resampler = torchaudio.transforms.Resample(orig_freq= sr, new_freq= target_sr)
        waveform_rs = resampler(waveform)
        return waveform_rs
    else:
        return waveform

In [13]:
def fix_length(max_len, waveform_rs):
    T = waveform_rs.shape[1]
    if T > max_len:
        waveform_fx = waveform_rs[:,:max_len]
    else:
        waveform_fx = torch.nn.functional.pad(waveform_rs, (0, max_len - T))
    return waveform_fx

In [14]:
def log_mel(freq_set, waveform_fx):
    mel = torchaudio.transforms.MelSpectrogram(
        sample_rate = freq_set,
        n_fft = 1024,
        hop_length= 256,
        n_mels= 64
    )
    spec = mel(waveform_fx)
    log_spec = torch.log(spec + 1e-9)
    return log_spec

In [15]:
def normalize(log_spec):
    return (log_spec - log_spec.mean()) / (log_spec.std() + 1e-6)    

In [16]:
time_max = 4 # secondes
freq_set = 16000 

In [17]:
row = df_train.iloc[7]
filepath = row['Filepath']
waveform, sr = extract_waveform(filepath)
waveform_rs = resample_waveform(sr, freq_set, waveform)
waveform_fx = fix_length(freq_set *time_max, waveform_rs)
log_spec = log_mel(freq_set, waveform_fx)
x = normalize(log_spec)
print("After resample:", waveform_rs.shape, "SR:", freq_set)
print("After pad/crop:", waveform_fx.shape, "seconds:", time_max)
print("Log-mel shape:", log_spec.shape)
print("Normalized mean/std:", float(x.mean()), float(x.std()))  
print('='*80)  
print(x.shape)  
print(x)  
print('='*80)  

After resample: torch.Size([1, 58726]) SR: 16000
After pad/crop: torch.Size([1, 64000]) seconds: 4
Log-mel shape: torch.Size([1, 64, 251])
Normalized mean/std: 1.2300878893256595e-07 0.9999998211860657
torch.Size([1, 64, 251])
tensor([[[-0.7830, -0.3612, -0.0080,  ..., -1.6689, -1.6689, -1.6689],
         [-0.9385, -0.4870, -0.0640,  ..., -1.6689, -1.6689, -1.6689],
         [-0.8114, -0.6811, -0.3712,  ..., -1.6689, -1.6689, -1.6689],
         ...,
         [-0.9970, -1.0327, -0.7330,  ..., -1.6689, -1.6689, -1.6689],
         [-1.0663, -1.0191, -0.7416,  ..., -1.6689, -1.6689, -1.6689],
         [-1.1315, -1.1536, -0.9042,  ..., -1.6689, -1.6689, -1.6689]]])


In [18]:
def process_data(filepath):
    waveform, sr = extract_waveform(filepath)
    waveform_rs = resample_waveform(sr, freq_set, waveform)
    waveform_fx = fix_length(freq_set *time_max, waveform_rs)
    log_spec = log_mel(freq_set, waveform_fx)
    x = normalize(log_spec)
    return x

In [19]:
test = process_data(filepath)
print('='*80)
print(filepath)
print('='*80)
print(test)

data\Train\03-01-07-01-01-01-11.wav
tensor([[[-0.7830, -0.3612, -0.0080,  ..., -1.6689, -1.6689, -1.6689],
         [-0.9385, -0.4870, -0.0640,  ..., -1.6689, -1.6689, -1.6689],
         [-0.8114, -0.6811, -0.3712,  ..., -1.6689, -1.6689, -1.6689],
         ...,
         [-0.9970, -1.0327, -0.7330,  ..., -1.6689, -1.6689, -1.6689],
         [-1.0663, -1.0191, -0.7416,  ..., -1.6689, -1.6689, -1.6689],
         [-1.1315, -1.1536, -0.9042,  ..., -1.6689, -1.6689, -1.6689]]])


In [20]:
min_lab = df_train["label_id"].min()
print("min label:", min_lab, "max label:", df_train["label_id"].max())

# if labels start at 1, shift them to start at 0
df_train = df_train.copy()
df_test  = df_test.copy()

df_train["label_id"] = df_train["label_id"] - min_lab
df_test["label_id"]  = df_test["label_id"]  - min_lab

NUM_CLASSES = df_train["label_id"].nunique()
print("NUM_CLASSES:", NUM_CLASSES)
print("train label range:", df_train["label_id"].min(), df_train["label_id"].max())


min label: 1 max label: 8
NUM_CLASSES: 8
train label range: 0 7


In [21]:
from torch.utils.data import  Dataset, DataLoader

In [22]:
class SERDataset(Dataset):
    def __init__(self, df):
        self.df = df.reset_index(drop= True)

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        x = process_data(row['Filepath'])
        y = int(row['label_id'])
        return x, y

In [23]:
counts = df_train['label_id'].value_counts().sort_index()
num_classes = df_train['label_id'].nunique()
counts = counts.reindex(range(num_classes), fill_value=0)

weights = 1.0 / (counts.values + 1e-9)
weights = weights / weights.sum() * num_classes

In [24]:
df_train['label_id'].nunique()

8

In [25]:
train_ds = SERDataset(df_train)
test_ds = SERDataset(df_test)

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True, num_workers=0)
test_loader  = DataLoader(test_ds, batch_size=32, shuffle=False, num_workers=0)

In [26]:
xb, yb = next(iter(train_loader))
print("X batch:", xb.shape)   # expected: [32, 1, 64, 251]
print("y batch:", yb.shape)   # expected: [32]
print("y example:", yb[:10])


X batch: torch.Size([32, 1, 64, 251])
y batch: torch.Size([32])
y example: tensor([3, 2, 3, 2, 5, 6, 5, 2, 6, 2])


In [27]:
import torch.nn as nn
import torch.nn.functional as F

In [28]:
df_train['label_id'].nunique()

8

In [29]:
NUM_CLASSES = df_train['label_id'].nunique()

class SimpleSER_CNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(1,32,3,padding= 1),
            nn.BatchNorm2d(32),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.2)
        )
        self.block2 = nn.Sequential(
            nn.Conv2d(32,64,3,padding= 1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Dropout(0.2)
        )
        self.block3 = nn.Sequential(
            nn.Conv2d(64,128,3,padding= 1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1,1))
        )
        
        self.fc = nn.Linear(128, num_classes)
    
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x).squeeze(-1).squeeze(-1)
        return self.fc(x)

In [30]:
import torch
print(torch.__version__)
print(torch.cuda.is_available())
print(torch.version.cuda)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")


2.5.1+cu121
True
12.1
NVIDIA GeForce RTX 4050 Laptop GPU


In [31]:
device = 'cuda' if torch.cuda.is_available() else "cpu"
print(f"Running on device: {device}")

weights = torch.tensor(weights, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=weights)

print("Class counts:\n", counts)
print("Class weights:\n", weights.detach().cpu().numpy())

model = SimpleSER_CNN(NUM_CLASSES).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr= 1e-3)

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode="min", factor=0.5, patience=3
)


EPOCHS = 60

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0
    for xb, yb in train_loader:

        xb, yb = xb.to(device), yb.to(device)
        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
    print(f"Epoch {epoch+1}/{EPOCHS} | train loss = {total_loss/len(train_loader):.4f}")
    avg_loss = total_loss / len(train_loader)
    scheduler.step(avg_loss)


Running on device: cuda
Class counts:
 label_id
0     62
1    123
2    123
3    123
4    123
5    122
6    123
7    122
Name: count, dtype: int64
Class weights:
 [1.7633891  0.8888628  0.8888628  0.8888628  0.8888628  0.89614856
 0.8888628  0.89614856]
Epoch 1/60 | train loss = 2.0122
Epoch 2/60 | train loss = 1.9358
Epoch 3/60 | train loss = 1.8620
Epoch 4/60 | train loss = 1.7993
Epoch 5/60 | train loss = 1.7415
Epoch 6/60 | train loss = 1.6817
Epoch 7/60 | train loss = 1.6288
Epoch 8/60 | train loss = 1.5933
Epoch 9/60 | train loss = 1.5614
Epoch 10/60 | train loss = 1.5539
Epoch 11/60 | train loss = 1.5084
Epoch 12/60 | train loss = 1.4912
Epoch 13/60 | train loss = 1.4708
Epoch 14/60 | train loss = 1.4403
Epoch 15/60 | train loss = 1.4236
Epoch 16/60 | train loss = 1.3997
Epoch 17/60 | train loss = 1.4118
Epoch 18/60 | train loss = 1.3738
Epoch 19/60 | train loss = 1.3739
Epoch 20/60 | train loss = 1.3182
Epoch 21/60 | train loss = 1.3080
Epoch 22/60 | train loss = 1.3036
Epoch 23

In [32]:
model.eval()
correct = total = 0

with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        preds = model(xb).argmax(dim=1)
        correct += (preds == yb).sum().item()
        total += yb.size(0)

print("Validation accuracy:", correct / total)

Validation accuracy: 0.4935064935064935


In [33]:
from sklearn.metrics import f1_score
import numpy as np

model.eval()
y_true = []
y_pred = []

with torch.no_grad():
    for xb, yb in test_loader:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        preds = logits.argmax(dim=1)

        y_true.extend(yb.cpu().numpy())
        y_pred.extend(preds.cpu().numpy())

y_true = np.array(y_true)
y_pred = np.array(y_pred)

macro_f1 = f1_score(y_true, y_pred, average="macro")
weighted_f1 = f1_score(y_true, y_pred, average="weighted")

print("Macro F1:", macro_f1)
print("Weighted F1:", weighted_f1)


Macro F1: 0.4675682808310585
Weighted F1: 0.4736314517843513


In [43]:
def eval_f1(loader):
    model.eval()
    yt, yp = [], []
    with torch.no_grad():
        for xb, yb in loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb).argmax(1)
            yt.extend(yb.cpu().numpy())
            yp.extend(preds.cpu().numpy())
    yt, yp = np.array(yt), np.array(yp)
    return f1_score(yt, yp, average="macro"), f1_score(yt, yp, average="weighted")

print("Train macro/weighted F1:", eval_f1(train_loader))
print("Val   macro/weighted F1:", eval_f1(test_loader))

Train macro/weighted F1: (0.5979155308197128, 0.5923128850861433)
Val   macro/weighted F1: (0.4675682808310585, 0.4736314517843513)


# Tests

In [34]:
time_max = 4 # secondes
freq_set = 16000

In [35]:
wv_rs =resample_waveform(sr, 16000, waveform)

In [36]:
waveform.shape

torch.Size([1, 176176])

In [37]:
wv_rs.shape[1]

58726

In [38]:
wv_rs.shape[1]/16000

3.670375

In [39]:
freq_set*time_max

64000

In [40]:
waveform_fx = fix_length(freq_set*time_max, wv_rs)

In [41]:
print(waveform_fx)

tensor([[0., 0., 0.,  ..., 0., 0., 0.]])


In [42]:
log_mel(freq_set, waveform_fx).shape

torch.Size([1, 64, 251])